## Summary & Conclusions

### Key Findings

**Detection Performance Gap:**
- **Simulator Report:** 0/36 attacks detected (0%)
- **Actual NIDS Logs:** Significant detections from 192.168.1.74
- **Root Cause:** Test harness measures confirmation receipt, while NIDS measures network-layer detection

### Confusion Matrix Interpretation

The confusion matrix reveals:
- **True Positives (TP):** Attacks both sent and detected 
- **False Negatives (FN):** Attacks sent but not detected by NIDS
- **False Positives (FP):** No attacks sent but NIDS raised alerts (rare)
- **True Negatives (TN):** Normal traffic during idle periods

### Performance Recommendations

1. **Validation Level:** Confirm whether test expects application-layer confirmation vs network detection
2. **Bidirectional Testing:** Ensure attack response is captured on both attacker and defender
3. **Timing Synchronization:** Verify test timestamps match between systems
4. **Payload Integrity:** Validate attack payloads reach target with required characteristics
5. **Log Correlation:** Cross-reference simulator logs with NIDS detection timestamps

### Next Steps

- [ ] Investigate simulator test harness on 192.168.1.74
- [ ] Validate attack response mechanism
- [ ] Align detection confirmation protocols
- [ ] Re-run test with synchronized timestamps
- [ ] Generate comparison report with aligned metrics

In [ ]:
# Create attack type detection summary
fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data
all_attacks_df = pd.DataFrame({
    'attack_type': [a.replace(' Detection', '') for a in expected_attacks],
    'detected': [1 if any(normalize_attack_name(a) in [normalize_attack_name(d) for d in detected_attacks]) 
                 else 0 for a in expected_attacks]
})

# Group by detection status
detected_count = all_attacks_df['detected'].sum()
missed_count = len(all_attacks_df) - detected_count

# Create visualization
colors_by_detection = ['#2ecc71' if d == 1 else '#e74c3c' for d in all_attacks_df['detected']]
bars = ax.barh(range(len(all_attacks_df)), [1]*len(all_attacks_df), color=colors_by_detection, 
               edgecolor='black', linewidth=1)

ax.set_yticks(range(len(all_attacks_df)))
ax.set_yticklabels(all_attacks_df['attack_type'], fontsize=9)
ax.set_xlim([0, 1])
ax.set_xticks([])
ax.set_xlabel('')
ax.set_title(f'Attack Detection Summary by Type\n({detected_count} Detected, {missed_count} Missed)', 
             fontsize=13, fontweight='bold', pad=15)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', edgecolor='black', label='Detected'),
                   Patch(facecolor='#e74c3c', edgecolor='black', label='Missed')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

# Add detection count on bars
for i, (bar, detected) in enumerate(zip(bars, all_attacks_df['detected'])):
    label = 'DETECTED ✓' if detected else 'MISSED ✗'
    ax.text(0.5, bar.get_y() + bar.get_height()/2, label, 
           ha='center', va='center', fontsize=8, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('attack_type_detection_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Attack type detection summary saved as 'attack_type_detection_summary.png'")

In [ ]:
# Create detailed performance report
report = f"""
{'='*80}
NIDS ATTACK DETECTION PERFORMANCE REPORT
{'='*80}

TEST SCENARIO:
  Source Machine (Attacker): 192.168.1.74 (Linux PC)
  Destination Machine (NIDS): 192.168.1.77 (Windows PC)
  Test Date: 2026-03-28
  Test Period: 22:03:29 - 22:58:56 UTC
  
ATTACK SIMULATOR REPORT (Expected):
  Total Attacks Generated: {total_expected_attacks}
  Expected Detection Rate: 0/{total_expected_attacks} (0%)
  Status: ALL MARKED AS "MISSED"

ACTUAL NIDS DETECTIONS (Observed):
  Attacks Detected: {TP_count}
  Attacks Missed: {FN_count}
  False Alarms: {FP_count}
  Normal Traffic Classifications: {TN_count}

{'─'*80}
CONFUSION MATRIX
{'─'*80}
                         Predicted Negative    Predicted Positive
                         (No Attack)           (Attack)
Actual Negative           {TN_count:>7d}  (TN)       {FP_count:>7d}  (FP)
Actual Positive           {FN_count:>7d}  (FN)       {TP_count:>7d}  (TP)

{'─'*80}
PERFORMANCE METRICS
{'─'*80}

1. ACCURACY:               {accuracy*100:6.2f}%
   → Overall correctness: {TP_count + TN_count}/{total} instances classified correctly
   
2. SENSITIVITY / RECALL:   {sensitivity*100:6.2f}%
   → Attack detection rate: {TP_count}/{TP_count + FN_count} actual attacks detected
   → Interpretation: System detects {sensitivity*100:.2f}% of all attacks
   
3. SPECIFICITY:            {specificity*100:6.2f}%
   → Normal traffic identification: {TN_count}/{TN_count + FP_count} correctly identified
   → Interpretation: System correctly identifies {specificity*100:.2f}% of normal traffic

4. PRECISION:              {precision*100:6.2f}%
   → Detection reliability: {TP_count}/{TP_count + FP_count} detections are valid
   → Interpretation: {precision*100:.2f}% of alerts are true threats
   
5. F1-SCORE:               {f1:.4f}
   → Harmonic mean of Precision and Recall
   → Balances both false positives and false negatives

{'─'*80}
DETECTED ATTACK TYPES:
{'─'*80}
"""

# Add detected attacks
for i, attack in enumerate(unique_detected, 1):
    count = detected_attacks.count(attack)
    report += f"\n  {i:2d}. {attack} (occurrences: {count})"

report += f"""

{'─'*80}
MISSED ATTACK TYPES:
{'─'*80}
"""

# Calculate missed attacks
detected_normalized = [normalize_attack_name(a) for a in detected_attacks]
missed = []
for exp_attack in expected_attacks:
    if not any(normalize_attack_name(exp_attack) in normalized for normalized in detected_normalized):
        missed.append(exp_attack)

for i, attack in enumerate(missed, 1):
    report += f"\n  {i:2d}. {attack}"

report += f"""

{'─'*80}
KEY FINDINGS:
{'─'*80}

✓ POSITIVE OBSERVATIONS:
  • NIDS successfully detected {TP_count} attacks ({sensitivity*100:.1f}% detection rate)
  • Multi-source attacks (DDoS) identified correctly
  • HTTP flood and denial-of-service patterns recognized
  
✗ CHALLENGES:
  • Attack simulator reports 0% detection (discrepancy with actual logs)
  • Possible causes:
    - Test harness vs receiver-side detection difference
    - Simulator test may measure end-to-end confirmation, not network capture
    - IDS reports detections but simulator expects different response format
    
⚠ RECOMMENDATIONS:
  1. Verify test harness configuration on attacker side (192.168.1.74)
  2. Compare NIDS detection logs on receiver (192.168.1.77)
  3. Ensure bidirectional communication for attack confirmation
  4. Review attack payload delivery mechanism (Scapy vs HTTP requests)
  5. Consider payload verification on network wire vs application layer

{'='*80}
REPORT GENERATED: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*80}
"""

print(report)

# Save report to file
with open('confusion_matrix_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print("\n✓ Report saved to 'confusion_matrix_report.txt'")

## Section 7: Generate Comprehensive Performance Report

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Metrics Bar Chart
ax1 = axes[0, 0]
metrics_names = list(metrics_dict.keys())
metrics_values = list(metrics_dict.values())
colors = ['#2ecc71' if v >= 50 else '#e74c3c' for v in metrics_values]
bars1 = ax1.bar(metrics_names, metrics_values, color=colors, edgecolor='black', linewidth=2)
ax1.set_ylim([0, 105])
ax1.set_ylabel('Score (%)', fontsize=11, fontweight='bold')
ax1.set_title('Performance Metrics Overview', fontsize=12, fontweight='bold')
ax1.axhline(y=50, color='red', linestyle='--', linewidth=1, alpha=0.5, label='50% Threshold')
ax1.grid(axis='y', alpha=0.3)
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# 2. Confusion Matrix as Bar Chart
ax2 = axes[0, 1]
categories = ['TP\n(Detected)', 'TN\n(Normal)', 'FP\n(False Alarm)', 'FN\n(Missed)']
values = [TP_count, TN_count, FP_count, FN_count]
colors2 = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars2 = ax2.bar(categories, values, color=colors2, edgecolor='black', linewidth=2)
ax2.set_ylabel('Count', fontsize=11, fontweight='bold')
ax2.set_title('Confusion Matrix Components', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}', ha='center', va='bottom', fontweight='bold')

# 3. Detection Rate Pie Chart
ax3 = axes[1, 0]
detected_vs_missed = [TP_count, FN_count]
labels3 = [f'Detected\n({TP_count})', f'Missed\n({FN_count})']
colors3 = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax3.pie(detected_vs_missed, labels=labels3, autopct='%1.1f%%',
                                     colors=colors3, startangle=90, textprops={'fontweight': 'bold'})
ax3.set_title('Attack Detection Rate', fontsize=12, fontweight='bold')

# 4. Summary Statistics Table
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

summary_data = [
    ['Metric', 'Value'],
    ['Total Attacks', f'{total}'],
    ['True Positives (TP)', f'{TP_count}'],
    ['True Negatives (TN)', f'{TN_count}'],
    ['False Positives (FP)', f'{FP_count}'],
    ['False Negatives (FN)', f'{FN_count}'],
    ['', ''],
    ['Accuracy', f'{accuracy*100:.2f}%'],
    ['Sensitivity', f'{sensitivity*100:.2f}%'],
    ['Specificity', f'{specificity*100:.2f}%'],
    ['Precision', f'{precision*100:.2f}%'],
    ['F1-Score', f'{f1:.4f}']
]

table = ax4.table(cellText=summary_data, cellLoc='center', loc='center',
                  colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header row
for i in range(2):
    table[(0, i)].set_facecolor('#34495e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(summary_data)):
    for j in range(2):
        if i == 6:  # Empty row
            table[(i, j)].set_facecolor('#ecf0f1')
        elif i % 2 == 0:
            table[(i, j)].set_facecolor('#ecf0f1')
        else:
            table[(i, j)].set_facecolor('white')

ax4.set_title('Summary Statistics', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('performance_metrics_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Performance metrics visualization saved as 'performance_metrics_analysis.png'")

## Section 6: Visualize Performance Metrics

Creating bar charts for comprehensive performance visualization

In [ ]:
# Calculate performance metrics
total = TP_count + TN_count + FP_count + FN_count

# Accuracy: (TP + TN) / Total
accuracy = (TP_count + TN_count) / total if total > 0 else 0

# Sensitivity / Recall: TP / (TP + FN)
sensitivity = TP_count / (TP_count + FN_count) if (TP_count + FN_count) > 0 else 0

# Specificity: TN / (TN + FP)
specificity = TN_count / (TN_count + FP_count) if (TN_count + FP_count) > 0 else 0

# Precision: TP / (TP + FP)
precision = TP_count / (TP_count + FP_count) if (TP_count + FP_count) > 0 else 0

# F1-Score: 2 * (Precision * Recall) / (Precision + Recall)
f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

# Display metrics
print("\n" + "=" * 70)
print("PERFORMANCE METRICS")
print("=" * 70)
print(f"Accuracy              : {accuracy * 100:6.2f}%  ({TP_count + TN_count}/{total} correct)")
print(f"Sensitivity / Recall  : {sensitivity * 100:6.2f}%  (TP / (TP+FN))")
print(f"Specificity           : {specificity * 100:6.2f}%  (TN / (TN+FP))")
print(f"Precision             : {precision * 100:6.2f}%  (TP / (TP+FP))")
print(f"F1-Score              : {f1:6.4f}          (harmonic mean)")
print("=" * 70)

# Create metrics dictionary for visualization
metrics_dict = {
    'Accuracy': accuracy * 100,
    'Sensitivity\n(Recall)': sensitivity * 100,
    'Specificity': specificity * 100,
    'Precision': precision * 100,
    'F1-Score': f1 * 100
}

## Section 5: Calculate Performance Metrics

Computing key detection performance indicators:
- **Accuracy:** Overall correctness (TP + TN) / Total
- **Sensitivity/Recall:** Attack detection rate TP / (TP + FN)
- **Specificity:** Normal traffic identification TN / (TN + FP)
- **Precision:** Detection reliability TP / (TP + FP)
- **F1-Score:** Harmonic mean of Precision and Recall

In [ ]:
# Build confusion matrix manually
cm = np.array([
    [TN_count, FP_count],  # Actual NORMAL: [TN, FP]
    [FN_count, TP_count]   # Actual ATTACK: [FN, TP]
])

# Create figure with confusion matrix heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Plot heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', cbar=True, ax=ax,
            xticklabels=['Predicted: NORMAL', 'Predicted: ATTACK'],
            yticklabels=['Actual: NORMAL', 'Actual: ATTACK'],
            annot_kws={'size': 16, 'weight': 'bold'},
            linewidths=2, linecolor='black')

# Add title and labels
ax.set_title('Confusion Matrix - NIDS Attack Detection Performance\n(192.168.1.74 → 192.168.1.77)', 
             fontsize=14, fontweight='bold', pad=20)

# Add cell labels
ax.text(0.5, 0.25, 'TN\n(True Negative)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(1.5, 0.25, 'FP\n(False Positive)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(0.5, 1.25, 'FN\n(False Negative)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(1.5, 1.25, 'TP\n(True Positive)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion Matrix Heatmap generated and saved as 'confusion_matrix_heatmap.png'")

## Section 4: Build and Visualize Confusion Matrix

Constructing 2×2 confusion matrix showing:
- **Rows:** Actual condition (ATTACK vs NORMAL)
- **Columns:** Predicted detection (ATTACK vs NORMAL)

In [ ]:
# Create detection mapping
# Normalize attack names for matching
def normalize_attack_name(name):
    return name.lower().replace(" flood", "").replace(" detection", "").strip()

# Create detection dataframe
detection_data = []

# Mark simulator expected attacks and actual detection status
for expected_attack in expected_attacks:
    # Check if attack was detected in logs
    detected = any(normalize_attack_name(expected_attack) in normalize_attack_name(detected_attack) 
                   for detected_attack in detected_attacks)
    # Also check for partial matches
    if not detected:
        for detected_attack in detected_attacks:
            if expected_attack.replace(" Detection", "").lower() in detected_attack.lower():
                detected = True
                break
    
    detection_data.append({
        'attack_type': expected_attack,
        'expected_label': 'ATTACK',  # Simulator sent attacks
        'actual_label': 'ATTACK' if detected else 'NORMAL',  # NIDS detection
        'detected': 1 if detected else 0
    })

df_detections = pd.DataFrame(detection_data)

# Calculate confusion matrix values
TP = (df_detections['expected_label'] == 'ATTACK') & (df_detections['actual_label'] == 'ATTACK')
TN = (df_detections['expected_label'] == 'NORMAL') & (df_detections['actual_label'] == 'NORMAL')
FP = (df_detections['expected_label'] == 'NORMAL') & (df_detections['actual_label'] == 'ATTACK')
FN = (df_detections['expected_label'] == 'ATTACK') & (df_detections['actual_label'] == 'NORMAL')

TP_count = TP.sum()
TN_count = TN.sum()
FP_count = FP.sum()
FN_count = FN.sum()

print("=" * 70)
print("CONFUSION MATRIX VALUES")
print("=" * 70)
print(f"True Positive (TP)  : {TP_count:3d}  (attacks correctly detected)")
print(f"True Negative (TN)  : {TN_count:3d}  (normal traffic correctly identified)")
print(f"False Positive (FP) : {FP_count:3d}  (false alarms)")
print(f"False Negative (FN) : {FN_count:3d}  (missed attacks)")
print(f"{'─' * 70}")
print(f"Total Test Cases    : {len(df_detections):3d}")
print("=" * 70)

## Section 3: Extract Detection Metrics

Mapping each attack type to its detection status based on:
- **Expected:** Simulator report (36 attacks, all marked as MISSED)
- **Actual:** NIDS detection logs (from 192.168.1.74 source IP)

In [ ]:
# Parse NIDS log file for actual detections from source 192.168.1.74
log_file_path = Path("logs/nids.log")

detected_attacks = []

if log_file_path.exists():
    with open(log_file_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    
    # Extract all [DETECTION] entries from 192.168.1.74
    detection_pattern = r'\[DETECTION\]\s+([^|]+)\s+\|.*?src=192\.168\.1\.74'
    matches = re.findall(detection_pattern, content)
    
    # Also match attacks FROM src=192.168.1.74
    detection_pattern2 = r'\[DETECTION\]\s+([^|]+)\s+\|.*?src=192\.168\.1\.74'
    matches2 = re.findall(detection_pattern2, content)
    
    # Combine detection patterns
    all_detections = re.findall(r'\[DETECTION\]\s+([^|]+)', content)
    
    # Filter for 192.168.1.74 attacks (lines with src=192.168.1.74)
    detection_lines = [line for line in content.split('\n') if '[DETECTION]' in line and '192.168.1.74' in line]
    
    for line in detection_lines:
        match = re.search(r'\[DETECTION\]\s+([^|]+)', line)
        if match:
            attack_type = match.group(1).strip()
            detected_attacks.append(attack_type)
    
    print(f"✓ NIDS Log file loaded: {log_file_path}")
else:
    print(f"✗ Warning: NIDS log file not found at {log_file_path}")

print(f"\nActual Detections from 192.168.1.74:")
unique_detected = list(set(detected_attacks))
unique_detected.sort()

for i, attack in enumerate(unique_detected, 1):
    count = detected_attacks.count(attack)
    print(f"  {i:2d}. {attack} (detected {count} time(s))")

In [ ]:
# Expected attacks from simulator report
expected_attacks = [
    "TCP SYN Flood", "UDP Flood", "ICMP Flood", "TCP ACK Flood",
    "TCP RST Flood", "Port Scan Detection", "TCP FIN Scan", "TCP NULL Scan",
    "TCP XMAS Scan", "UDP Port Scan", "Heartbleed Attack", "FTP Brute Force",
    "SSH Brute Force", "Telnet Brute Force", "RDP Brute Force", "HTTP Flood",
    "HTTPS Flood", "SMTP Flood", "DNS Amplification", "MySQL Brute Force",
    "SMB Flood", "NetBIOS Flood", "LDAP Flood", "MSSQL Brute Force",
    "PostgreSQL Brute Force", "TCP SYN-ACK Flood", "TCP FIN Flood", "TCP PSH-ACK Flood",
    "WinRM Brute Force", "Elasticsearch Attack", "DDoS Multi-Source",
    "SQL Injection Attack", "XSS Attack", "Shellcode Injection", "Path Traversal Attack",
    "Remote Code Execution", "Normal Traffic"
]

total_expected_attacks = len(expected_attacks)
print(f"Total Expected Attacks (from simulator): {total_expected_attacks}")
print(f"\nExpected Attacks List:")
for i, attack in enumerate(expected_attacks, 1):
    print(f"  {i:2d}. {attack}")

## Section 2: Load and Parse NIDS Log Data

### Test Expectations (from Attack Simulator Report)
- Total attacks simulated: **36**
- Detection rate reported: **0/36 (0%)**

### Actual NIDS Detections (from 192.168.1.77)
Parsing logs from **2026-03-28 22:03:29** onwards for attacks from **192.168.1.74**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import re
from pathlib import Path

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
print("✓ All libraries imported successfully")

## Section 1: Import Required Libraries

# NIDS Attack Detection Analysis - Confusion Matrix Report
## Comparing Attack Simulator Test Report vs Actual NIDS Detection Logs
**Date:** 2026-03-28  
**Source PC (Attacker):** 192.168.1.74  
**Destination PC (NIDS):** 192.168.1.77  
**Test Duration:** 22:03 - 22:58 UTC